# SWARMRoute — Evaluation Notebook

This notebook integrates `src/evaluation/plotting.py` (Matplotlib), benchmark comparisons, contract-net mesh bidding metrics, and the real-time weather provider.

Run from the repo root (`jupyter lab` or `jupyter notebook`), with this file located in `notebooks/`.


In [ ]:
import sys
import json
from pathlib import Path

# Make the repo root importable when running from notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
%matplotlib inline

## 1. Load existing plotting utilities

Imports evaluation plotting functions from `src.evaluation.plotting`.

In [ ]:
try:
    from src.evaluation import plotting
    print("Loaded plotting utilities:", [f for f in dir(plotting) if not f.startswith("_")])
except ImportError as e:
    print("Could not import src.evaluation.plotting — check REPO_ROOT / package layout.")
    print(e)

## 2. PPO vs OR-Tools baseline comparison

Loads real benchmark evaluation results from `results/benchmarks/ppo_comparison.json` (or `results/experiments/ppo_ablation.json`) to visualize performance across PPO Adaptive agents and baselines.

In [ ]:
comparison_file = REPO_ROOT / "results" / "benchmarks" / "ppo_comparison.json"
if comparison_file.exists():
    with open(comparison_file, "r") as f:
        benchmarks = json.load(f)
    methods = list(benchmarks.keys())
    success_rates = [benchmarks[m]["delivery_success_pct"] for m in methods]
    on_time_rates = [benchmarks[m]["on_time_delivery_pct"] for m in methods]
    
    fig, ax = plt.subplots(figsize=(9, 5))
    x = range(len(methods))
    width = 0.35
    ax.bar([i - width / 2 for i in x], success_rates, width, label="Delivery Success %", color="#2563eb")
    ax.bar([i + width / 2 for i in x], on_time_rates, width, label="On-Time Delivery %", color="#10b981")
    ax.set_xticks(list(x))
    ax.set_xticklabels(methods, rotation=20, ha="right")
    ax.set_ylabel("Percentage (%)")
    ax.set_title("Benchmark Comparison: PPO Adaptive vs Baselines")
    ax.legend()
    ax.grid(True, axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    example = {"OR-Tools": 85.2, "PPO Config A": 92.6, "PPO Adaptive": 85.2}
    plt.figure(figsize=(6, 4))
    plt.bar(example.keys(), example.values(), color="#2563eb")
    plt.ylabel("Delivery Success %")
    plt.title("PPO vs OR-Tools baseline (fallback)")
    plt.tight_layout()
    plt.show()

## 3. Mesh / contract-net bidding activity

Visualizes multi-hop mesh network latency and bidding phase timings from `results/experiments/flagship_recovery.json`.

In [ ]:
flagship_file = REPO_ROOT / "results" / "experiments" / "flagship_recovery.json"
if flagship_file.exists():
    with open(flagship_file, "r") as f:
        flagship = json.load(f)
    timings = flagship.get("timings", {})
    ms_keys = [k for k in timings.keys() if "ms" in k]
    step_names = [k.replace("_ms", "").replace("_", " ").title() for k in ms_keys]
    step_vals = [timings[k] for k in ms_keys]
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(step_names, step_vals, color="#8b5cf6")
    ax.set_xlabel("Latency (ms)")
    ax.set_title("Contract-Net Mesh Bidding & Recovery Latency Stages")
    ax.grid(True, axis="x", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    from src.networking.mesh import MeshNetwork
    mesh = MeshNetwork(transmission_range_km=25.0, seed=42)
    print("Mesh initialized with range:", mesh.transmission_range_km)

## 4. Real-Time Weather Module Demonstration

Demonstrates `src/data/weather.py` with mock and live provider options, condition severity scoring, and speed multiplier modulation.

In [ ]:
from src.data.weather import MockWeatherProvider, WeatherCondition

for condition in [WeatherCondition.CLEAR, WeatherCondition.CLOUDY, WeatherCondition.RAIN, WeatherCondition.HEAVY_RAIN, WeatherCondition.STORM]:
    provider = MockWeatherProvider(seed=42, bias=condition)
    snap = provider.get_current(12.9716, 77.5946)
    print(f"Condition: {snap.condition.value:11s} | Severity: {snap.severity:.2f} | Speed Multiplier: {snap.speed_multiplier:.2f} | Temp: {snap.temperature_c}°C")